# Ting – Aggregazione e statistiche multi-campione

Legge i `ting_batch_summary.csv` di più cartelle, li unisce raggruppando per etichetta assegnata manualmente, e produce:
- **PDF** con boxplot (E0, betaE, tc, R²) e tabella medie ± SD per gruppo
- **CSV** con tutte le statistiche aggregate

> ⚠️ I CSV devono essere stati generati con la versione **aggiornata** di `ting_utils.py`  
> (quella che include le colonne `E0_Pa`, `betaE`, `tc_ms` ecc.).

In [62]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
print('OK')

OK


## ⚙️ CONFIGURAZIONE — modifica solo questa cella

In [63]:
# Scoperta automatica delle cartelle da unire sotto ANALISI/CONCORR/UMANE
# Ogni sottocartella data puo contenere le tipologie A1 e/o 0R
from pathlib import Path

CONCORR_UMANE_ROOT = Path("/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO")
INCLUDE_TYPES = ("A1", "0R")

FOLDERS = []
for date_dir in sorted(CONCORR_UMANE_ROOT.iterdir()):
    if not date_dir.is_dir():
        continue
    for sample_type in INCLUDE_TYPES:
        sample_dir = date_dir / sample_type
        if sample_dir.is_dir():
            FOLDERS.append({
                "path": str(sample_dir),
                "label": f"{date_dir.name}_{sample_type}",
            })

print(f"Cartelle rilevate automaticamente: {len(FOLDERS)}")
for entry in FOLDERS:
    print(f"  {entry['label']}: {entry['path']}")
# Dove salvare l'output
OUTPUT_DIR = "/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated"
# Filtra solo curve con status == 'ok'
ONLY_OK = True
# Escludi curve con betaE fissato al bound (fit potenzialmente degenere)
EXCLUDE_BETA_AT_BOUND = True
# Escludi curve con tempo tc negativo (fit non fisico / degenerato)
EXCLUDE_NEGATIVE_TC = True
# Escludi curve con R^2 negativo (fit peggiore della media)
EXCLUDE_NEGATIVE_R2 = True
R2_COLUMN_FOR_FILTER = "r2_plr"
# Abilita aggregazione trasversale: tutte le A1 insieme e tutte le 0R insieme
ENABLE_A1_0R_AGGREGATION = True

# Filtro robusto per curve estremamente fuori distribuzione (MAD-zscore)
ENABLE_EXTREME_CURVE_FILTER = True
EXTREME_FILTER_TARGET = "larger"  # "larger" | "A1" | "0R" | "all"
EXTREME_FILTER_COLUMNS = ("E0_Pa", "betaE", "tc_ms", "r2_plr")
EXTREME_FILTER_MAD_Z = 4.5
EXTREME_FILTER_MAX_FRACTION = 0.20
EXTREME_FILTER_MIN_CURVES_PER_MACRO = 60

# Nei boxplot: un punto per ogni curva (non media per cellula)
PLOT_EACH_CURVE_POINT = True
# Nei boxplot: scrivi il nome della cellula vicino a ciascun punto (se disponibile)
LABEL_POINTS_WITH_CELL_NAME = False
# Parametri da includere nel riassunto — chiave: nome colonna CSV, valore: etichetta leggibile
PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
    "tc_ms":          "tc [ms]",
    "r2_plr":         "R² PLR",
}
TABLE_PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
    "tc_ms":          "tc [ms]",
    "r2_plr":         "R² PLR",
}
PLOT_PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output in: {OUTPUT_DIR}")

Cartelle rilevate automaticamente: 8
  100426_A1: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/100426/A1
  140326_A1: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/140326/A1
  140326_0R: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/140326/0R
  150426_0R: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/150426/0R
  160426_0R: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/160426/0R
  240326_A1: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/240326/A1
  240326_0R: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/240326/0R
  270326_A1: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/270326/A1
Output in: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated


## 1. Scoperta automatica dei CSV

In [64]:
csv_entries = []
for entry in FOLDERS:
    p = os.path.join(entry["path"], "ting_batch_summary.csv")
    if not os.path.isfile(p):
        print(f"[MANCANTE] {p}")
    else:
        csv_entries.append({"path": p, "label": entry["label"]})
        print(f"  OK  [{entry['label']}]  {p}")

if not csv_entries:
    raise FileNotFoundError("Nessun ting_batch_summary.csv trovato. Controlla i percorsi in FOLDERS.")

print(f"\n{len(csv_entries)} cartelle pronte.")

  OK  [100426_A1]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/100426/A1/ting_batch_summary.csv
  OK  [140326_A1]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/140326/A1/ting_batch_summary.csv
  OK  [140326_0R]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/140326/0R/ting_batch_summary.csv
  OK  [150426_0R]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/150426/0R/ting_batch_summary.csv
  OK  [160426_0R]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/160426/0R/ting_batch_summary.csv
  OK  [240326_A1]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/240326/A1/ting_batch_summary.csv
  OK  [240326_0R]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/240326/0R/ting_batch_summary.csv
  OK  [270326_A1]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/CONCORR/TOPO/270326/A1/ting_ba

## 2. Caricamento e pulizia

In [65]:
dfs = []

for entry in csv_entries:

    df_tmp = pd.read_csv(entry["path"])
    df_tmp["group"] = entry["label"]
    df_tmp["source_path"] = entry["path"]

    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

print(f"Curve totali caricate: {len(df)}")

# Filtri qualità
if ONLY_OK:
    before = len(df)
    df = df[df["status"].astype(str) == "ok"]
    print(f"Dopo filtro status==ok: {before} → {len(df)}")

if EXCLUDE_BETA_AT_BOUND and "betaE_at_bound" in df.columns:
    before = len(df)
    df = df[df["betaE_at_bound"] != True]
    print(f"Dopo filtro betaE_at_bound: {before} → {len(df)}")

def infer_a1_0r_group_from_text(text):
    s = str(text).upper().replace("-", "_")
    if "_A1" in s or s.endswith("A1") or "A1_" in s or "/A1" in s:
        return "A1"
    if "_0R" in s or s.endswith("0R") or "0R_" in s or "/0R" in s:
        return "0R"
    return np.nan

agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
if agg_enabled:
    macro_from_label = df["group"].map(infer_a1_0r_group_from_text)
    macro_from_path = df["source_path"].map(infer_a1_0r_group_from_text) if "source_path" in df.columns else np.nan
    df["macro_group"] = macro_from_label.fillna(macro_from_path)

    macro_counts = df["macro_group"].value_counts(dropna=True)
    print("\nCurve aggregate A1/0R:")
    if len(macro_counts) == 0:
        print("  Nessuna curva riconosciuta come A1 o 0R dalle label/path.")
    else:
        for name, count in macro_counts.items():
            print(f"  {name}: {count}")

# Converti in numerico e controlla colonne disponibili
available = {}
for col, label in PARAMS.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        available[col] = label
    elif col == "young_hertz_pa" and "young_hertz_kpa" in df.columns:
        df["young_hertz_kpa"] = pd.to_numeric(df["young_hertz_kpa"], errors="coerce")
        df[col] = df["young_hertz_kpa"] * 1000.0
        available[col] = label
    else:
        print(f"  [mancante] {col} — riesegui il batch con ting_utils.py aggiornato")

if EXCLUDE_NEGATIVE_TC and "tc_ms" in df.columns:
    tc_vals = pd.to_numeric(df["tc_ms"], errors="coerce")
    neg_tc = tc_vals < 0
    if bool(neg_tc.any()):
        n_neg = int(neg_tc.sum())
        print(f"  [filtro] tc_ms negativi rimossi: {n_neg}")
        df = df.loc[~neg_tc].copy()

if EXCLUDE_NEGATIVE_R2 and R2_COLUMN_FOR_FILTER in df.columns:
    r2_vals = pd.to_numeric(df[R2_COLUMN_FOR_FILTER], errors="coerce")
    neg_r2 = r2_vals < 0
    if bool(neg_r2.any()):
        n_neg_r2 = int(neg_r2.sum())
        print(f"  [filtro] {R2_COLUMN_FOR_FILTER} negativi rimossi: {n_neg_r2}")
        df = df.loc[~neg_r2].copy()

# Filtro robusto curve estreme via MAD-zscore
if agg_enabled and bool(globals().get("ENABLE_EXTREME_CURVE_FILTER", False)) and "macro_group" in df.columns:
    cols_filter = [c for c in globals().get("EXTREME_FILTER_COLUMNS", ()) if c in df.columns]
    target_mode = str(globals().get("EXTREME_FILTER_TARGET", "larger")).lower()
    z_thr = float(globals().get("EXTREME_FILTER_MAD_Z", 4.5))
    max_fraction = float(globals().get("EXTREME_FILTER_MAX_FRACTION", 0.20))
    min_curves = int(globals().get("EXTREME_FILTER_MIN_CURVES_PER_MACRO", 60))

    macro_counts_now = df["macro_group"].value_counts(dropna=True)
    target_macros = []
    if len(macro_counts_now) > 0 and cols_filter:
        if target_mode == "larger":
            target_macros = [str(macro_counts_now.idxmax())]
        elif target_mode in ("a1", "0r"):
            target_macros = [target_mode.upper()]
        elif target_mode == "all":
            target_macros = [str(x) for x in macro_counts_now.index.tolist()]

    def _robust_z(vals):
        vals = np.asarray(vals, dtype=float)
        med = np.nanmedian(vals)
        mad = np.nanmedian(np.abs(vals - med))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-12:
            return np.zeros_like(vals, dtype=float)
        return (vals - med) / scale

    total_removed = 0
    for macro_name in target_macros:
        idx = df.index[df["macro_group"].astype(str) == macro_name]
        if len(idx) <= min_curves:
            continue

        sub = df.loc[idx, cols_filter].apply(pd.to_numeric, errors="coerce")
        z_cols = []
        for c in cols_filter:
            z = _robust_z(sub[c].values)
            z_cols.append(np.abs(z))
        if not z_cols:
            continue

        z_matrix = np.vstack(z_cols).T
        score = np.nanmax(z_matrix, axis=1)
        score = np.where(np.isfinite(score), score, -np.inf)

        cand_local = np.where(score > z_thr)[0]
        if cand_local.size == 0:
            continue

        max_remove_by_fraction = int(np.floor(max_fraction * len(idx)))
        max_remove_by_min = max(0, len(idx) - min_curves)
        max_remove = min(cand_local.size, max_remove_by_fraction, max_remove_by_min)
        if max_remove <= 0:
            continue

        order = np.argsort(score[cand_local])[::-1]
        to_remove_local = cand_local[order[:max_remove]]
        remove_idx = idx[to_remove_local]
        df = df.drop(index=remove_idx)
        total_removed += len(remove_idx)
        print(f"  [filtro estremi] {macro_name}: rimosse {len(remove_idx)} curve (thr>|z|>{z_thr}, colonne={cols_filter})")

    if total_removed == 0:
        print("  [filtro estremi] nessuna curva rimossa")
    else:
        print(f"  [filtro estremi] totale curve rimosse: {total_removed}")

print(f"\nParametri disponibili: {list(available.keys())}")
print(f"Gruppi trovati: {sorted(df['group'].unique())}")
if "macro_group" in df.columns:
    print("Macro-gruppi dopo filtri:", df["macro_group"].value_counts(dropna=True).to_dict())
df.head(3)

Curve totali caricate: 326
Dopo filtro status==ok: 326 → 326

Curve aggregate A1/0R:
  0R: 168
  A1: 158
  [filtro] tc_ms negativi rimossi: 4
  [filtro estremi] 0R: rimosse 28 curve (thr>|z|>4.5, colonne=['E0_Pa', 'betaE', 'tc_ms', 'r2_plr'])
  [filtro estremi] totale curve rimosse: 28

Parametri disponibili: ['E0_Pa', 'betaE', 'tc_ms', 'r2_plr']
Gruppi trovati: ['100426_A1', '140326_0R', '140326_A1', '150426_0R', '160426_0R', '240326_0R', '240326_A1', '270326_A1']
Macro-gruppi dopo filtri: {'A1': 155, '0R': 139}


,curve,cell_folder,fd_folder,path,status,selected_model,selection_reason,E0_Pa,betaE,tc_ms,...,tau2_ms,rmse_gm2_pN,r2_gm2,retrace_trim_points,closure_refined,contact_strategy,force_drag_applied,group,source_path,macro_group
0,cell1_FD-0000,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,PLR,BIC=-89198.19 | R2=0.9981 | scelta penalizzand...,1516.326475,0.054521,18.075387,...,NaN,NaN,NaN,0,True,current_default,False,100426_A1,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,A1
1,cell1_FD-0001,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,PLR,BIC=-88559.61 | R2=0.9943 | scelta penalizzand...,1275.283140,0.080530,15.967113,...,NaN,NaN,NaN,0,True,current_default,False,100426_A1,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,A1
2,cell1_FD-0002,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,PLR,BIC=-87342.55 | R2=0.9716 | scelta penalizzand...,1441.783525,0.050822,19.000000,...,NaN,NaN,NaN,0,False,current_default,False,100426_A1,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,A1


## 3. Statistiche per cellula + media globale

In [66]:
rows_stats = []
for group, gdf in df.groupby("group", sort=True):
    row = {"Cellula": group, "N curve": len(gdf)}
    for col, label in TABLE_PARAMS.items():
        vals = gdf[col].dropna()
        if len(vals) == 0:
            row[f"{label} media"] = None
            row[f"{label} SD"] = None
        else:
            row[f"{label} media"] = round(float(np.mean(vals)), 4)
            row[f"{label} SD"] = round(float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0, 4)
    rows_stats.append(row)

# Righe finali: medie globali separate per tipo (A1 e 0R)
agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
if agg_enabled and "macro_group" in df.columns:
    for macro_name in ["A1", "0R"]:
        sub = df[df["macro_group"] == macro_name]
        if len(sub) == 0:
            continue
        row_global = {"Cellula": f"─── MEDIA GLOBALE {macro_name} ───", "N curve": len(sub)}
        for col, label in TABLE_PARAMS.items():
            vals_all = sub[col].dropna()
            if len(vals_all) == 0:
                row_global[f"{label} media"] = None
                row_global[f"{label} SD"] = None
            else:
                row_global[f"{label} media"] = round(float(np.mean(vals_all)), 4)
                row_global[f"{label} SD"] = round(float(np.std(vals_all, ddof=1)) if len(vals_all) > 1 else 0.0, 4)
        rows_stats.append(row_global)

df_stats = pd.DataFrame(rows_stats)
df_stats


,Cellula,N curve,E₀ [Pa] media,E₀ [Pa] SD,βE media,βE SD,tc [ms] media,tc [ms] SD,R² PLR media,R² PLR SD
0,100426_A1,58,763.7742,304.4833,0.0718,0.0314,11.4664,4.6460,0.9915,0.0163
1,140326_0R,29,596.2277,383.9443,0.0654,0.0281,16.5783,4.3157,0.9896,0.0121
2,140326_A1,24,526.3741,201.7915,0.0870,0.0268,14.7434,3.3948,0.9964,0.0037
3,150426_0R,45,1138.6617,319.0149,0.1066,0.0452,15.3840,3.0153,0.9805,0.0212
4,160426_0R,29,939.0648,300.1829,0.1009,0.0440,14.2398,2.9663,0.9832,0.0212
5,240326_0R,36,1548.2275,594.7939,0.0880,0.0428,16.3186,2.5307,0.9786,0.0204
6,240326_A1,29,563.0634,230.0049,0.0939,0.0355,12.7850,6.2902,0.9814,0.0268
7,270326_A1,44,689.2994,200.6061,0.0663,0.0313,11.8294,3.7653,0.9928,0.0118
8,─── MEDIA GLOBALE A1 ───,155,668.3219,264.6417,0.0767,0.0329,12.3235,4.7077,0.9908,0.0171
9,─── MEDIA GLOBALE 0R ───,139,1089.9241,530.9793,0.0920,0.0437,15.6365,3.2903,0.9825,0.0197


## 4. Salvataggio CSV

In [67]:
path_stats = os.path.join(OUTPUT_DIR, "ting_summary_per_cellula_A1_0R.csv")
df_stats.to_csv(path_stats, index=False)
print(f"Statistiche per cellula: {path_stats}")

path_all = os.path.join(OUTPUT_DIR, "ting_summary_tutte_le_curve_A1_0R.csv")
df.to_csv(path_all, index=False)
print(f"Tutte le curve: {path_all}")

Statistiche per cellula: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_per_cellula_A1_0R.csv
Tutte le curve: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_tutte_le_curve_A1_0R.csv


## 5. PDF — tabella medie ± SD + boxplot per parametro

In [68]:
groups = sorted(df["group"].unique())
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors = {g: palette[i % len(palette)] for i, g in enumerate(groups)}

pdf_path = os.path.join(OUTPUT_DIR, "ting_summary_report_A1_0R.pdf")
np.random.seed(0)


def build_stats_table_df(df_in, available_params):
    rows = []
    for group, gdf in df_in.groupby("group", sort=True):
        row = {"Cellula": group, "N curve": len(gdf)}
        for col, label in available_params.items():
            vals = gdf[col].dropna()
            if len(vals) == 0:
                row[f"{label} media"] = None
                row[f"{label} SD"] = None
            else:
                row[f"{label} media"] = round(float(np.mean(vals)), 4)
                row[f"{label} SD"] = round(float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0, 4)
        rows.append(row)

    # Nessuna media globale combinata: solo globale per tipo A1/0R (se presenti)
    agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
    if agg_enabled and "macro_group" in df_in.columns:
        for macro_name in ["A1", "0R"]:
            sub = df_in[df_in["macro_group"] == macro_name]
            if len(sub) == 0:
                continue
            row_global = {"Cellula": f"─── MEDIA GLOBALE {macro_name} ───", "N curve": len(sub)}
            for col, label in available_params.items():
                vals_all = sub[col].dropna()
                if len(vals_all) == 0:
                    row_global[f"{label} media"] = None
                    row_global[f"{label} SD"] = None
                else:
                    row_global[f"{label} media"] = round(float(np.mean(vals_all)), 4)
                    row_global[f"{label} SD"] = round(float(np.std(vals_all, ddof=1)) if len(vals_all) > 1 else 0.0, 4)
            rows.append(row_global)

    return pd.DataFrame(rows)


def add_table_page(pdf, table_df, title):
    col_headers = list(table_df.columns)
    cell_text = [
        [
            str(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else "—"
            for v in row
        ]
        for row in table_df.itertuples(index=False)
    ]

    fig, ax = plt.subplots(figsize=(max(10, len(col_headers) * 1.8), max(3, len(table_df) * 0.5 + 1.5)))
    ax.axis("off")
    ax.set_title(title, fontsize=13, pad=14)
    tbl = ax.table(cellText=cell_text, colLabels=col_headers, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1.0, 1.5)
    tbl.auto_set_column_width(list(range(len(col_headers))))
    for (r, _), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#DDEEFF")
            cell.set_text_props(weight="bold")
        elif r > 0 and cell_text[r - 1][0].startswith("─"):
            cell.set_facecolor("#FFF3CD")
            cell.set_text_props(weight="bold")
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_boxplot_page(pdf, df_sub, groups_sub, colors_map, col, label, title):
    """Boxplot per un sottoinsieme di gruppi su una pagina PDF."""
    data = [df_sub[df_sub["group"] == g][col].dropna().values for g in groups_sub]
    has_cf = "cell_folder" in df_sub.columns

    fig, ax = plt.subplots(figsize=(max(6, len(groups_sub) * 1.5), 5))
    bp = ax.boxplot(
        data,
        patch_artist=True,
        widths=0.5,
        medianprops=dict(color="black", linewidth=2),
    )
    for patch, g in zip(bp["boxes"], groups_sub):
        patch.set_facecolor(colors_map[g])
        patch.set_alpha(0.6)

    for i, g in enumerate(groups_sub, start=1):
        cols_to_take = [col]
        if has_cf:
            cols_to_take.append("cell_folder")

        gdf = df_sub[df_sub["group"] == g][cols_to_take].dropna(subset=[col]).copy()
        curve_vals = gdf[col].values

        jitter = np.random.uniform(-0.15, 0.15, size=len(curve_vals))
        x_points = np.full(len(curve_vals), i) + jitter

        ax.scatter(
            x_points,
            curve_vals,
            color=colors_map[g],
            alpha=0.80,
            s=40,
            zorder=3,
            edgecolors="white",
            linewidths=0.5,
        )

        if LABEL_POINTS_WITH_CELL_NAME and has_cf:
            labels = gdf["cell_folder"].astype(str).values
            for x, y, name in zip(x_points, curve_vals, labels):
                ax.annotate(
                    name,
                    (x, y),
                    textcoords="offset points",
                    xytext=(3, 3),
                    fontsize=6,
                    alpha=0.75,
                )

    ax.set_xticks(range(1, len(groups_sub) + 1))
    ax.set_xticklabels(groups_sub, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


with PdfPages(pdf_path) as pdf:
    # 1) Tabella generale sempre presente nel PDF
    global_stats = build_stats_table_df(df, TABLE_PARAMS)
    add_table_page(
        pdf,
        global_stats,
        f"Riassunto Ting completo — {' | '.join(e['label'] for e in csv_entries)}"
    )

    # 2) Boxplot dei parametri richiesti
    for col, label in PLOT_PARAMS.items():
        add_boxplot_page(pdf, df, groups, colors, col, label, label)

    # 3) Aggiungi analisi separata A1 / 0R (se disponibile)
    agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
    if agg_enabled and "macro_group" in df.columns:
        any_macro = False
        for macro_name in ["A1", "0R"]:
            sub = df[df["macro_group"] == macro_name].copy()
            if len(sub) == 0:
                continue
            any_macro = True
            sub_stats = build_stats_table_df(sub, TABLE_PARAMS)
            add_table_page(
                pdf,
                sub_stats,
                f"Riassunto Ting {macro_name} — {' | '.join(e['label'] for e in csv_entries)}",
            )
            groups_sub = sorted(sub["group"].unique())
            for col, label in PLOT_PARAMS.items():
                if len(groups_sub) == 0:
                    continue
                add_boxplot_page(
                    pdf, sub, groups_sub, colors, col, label,
                    f"{label} — {macro_name}",
                )

        if not any_macro:
            print("[INFO] Nessuna curva classificata come A1/0R: nel PDF resta la tabella generale e i boxplot globali.")

        # 4) Grafico finale A1 vs 0R (E0 + betaE) se entrambe disponibili
        if any_macro and "E0_Pa" in df.columns and "betaE" in df.columns:
            macro_groups = [g for g in ["A1", "0R"] if (df["macro_group"] == g).any()]
            if len(macro_groups) > 0:
                e0_color = "#1f77b4"
                beta_color = "#d62728"

                fig, ax1 = plt.subplots(figsize=(8.2, 5.6))
                ax2 = ax1.twinx()

                e0_points_by_macro = {}
                beta_points_by_macro = {}
                mean_curve_e0 = {}
                mean_curve_beta = {}

                for g in macro_groups:
                    gdf_macro = df[df["macro_group"] == g]
                    e0_points = gdf_macro["E0_Pa"].dropna().values
                    beta_points = gdf_macro["betaE"].dropna().values

                    e0_points_by_macro[g] = e0_points
                    beta_points_by_macro[g] = beta_points
                    curve_e0 = gdf_macro["E0_Pa"].dropna().values
                    curve_beta = gdf_macro["betaE"].dropna().values
                    mean_curve_e0[g] = float(np.mean(curve_e0)) if len(curve_e0) > 0 else np.nan
                    mean_curve_beta[g] = float(np.mean(curve_beta)) if len(curve_beta) > 0 else np.nan

                x_base = np.arange(1, len(macro_groups) + 1)
                off = 0.16
                width = 0.25

                data_e0 = [e0_points_by_macro[g] for g in macro_groups]
                data_beta = [beta_points_by_macro[g] for g in macro_groups]

                bp_e0 = ax1.boxplot(
                    data_e0,
                    positions=x_base - off,
                    patch_artist=True,
                    widths=width,
                    medianprops=dict(color="none", linewidth=0),
                )
                for patch in bp_e0["boxes"]:
                    patch.set_facecolor(e0_color)
                    patch.set_alpha(0.28)
                    patch.set_edgecolor(e0_color)

                bp_beta = ax2.boxplot(
                    data_beta,
                    positions=x_base + off,
                    patch_artist=True,
                    widths=width,
                    medianprops=dict(color="none", linewidth=0),
                )
                for patch in bp_beta["boxes"]:
                    patch.set_facecolor(beta_color)
                    patch.set_alpha(0.22)
                    patch.set_edgecolor(beta_color)

                for i, g in enumerate(macro_groups, start=1):
                    y_e0 = e0_points_by_macro[g]
                    y_beta = beta_points_by_macro[g]
                    j_e0 = np.random.uniform(-0.06, 0.06, size=len(y_e0))
                    j_beta = np.random.uniform(-0.06, 0.06, size=len(y_beta))
                    ax1.scatter(
                        np.full(len(y_e0), i - off) + j_e0,
                        y_e0,
                        color=e0_color,
                        alpha=0.55,
                        s=20,
                        zorder=3,
                        edgecolors="white",
                        linewidths=0.4,
                    )
                    ax2.scatter(
                        np.full(len(y_beta), i + off) + j_beta,
                        y_beta,
                        color=beta_color,
                        alpha=0.55,
                        s=20,
                        zorder=3,
                        edgecolors="white",
                        linewidths=0.4,
                    )

                    if len(y_e0) > 0:
                        ax1.hlines(
                            mean_curve_e0[g],
                            i - off - 0.12,
                            i - off + 0.12,
                            colors="black",
                            linewidth=2.5,
                            zorder=4,
                        )
                    if len(y_beta) > 0:
                        ax2.hlines(
                            mean_curve_beta[g],
                            i + off - 0.12,
                            i + off + 0.12,
                            colors="black",
                            linewidth=2.5,
                            zorder=4,
                        )

                ax1.set_xlim(0.5, len(macro_groups) + 0.5)
                ax1.set_xticks(x_base)
                ax1.set_xticklabels(macro_groups, fontsize=10)
                ax1.set_ylabel("E₀ [Pa]", fontsize=11, color=e0_color)
                ax2.set_ylabel("βE [-]", fontsize=11, color=beta_color)
                ax1.tick_params(axis="y", labelcolor=e0_color)
                ax2.tick_params(axis="y", labelcolor=beta_color)
                ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))
                ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))
                ax1.set_title("Boxplot finale A1 vs 0R: E₀ e βE (doppia ordinata)", fontsize=13)
                ax1.grid(axis="y", linestyle="--", alpha=0.35)

                fig.tight_layout()
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

                print("\nGrafico finale E0 + betaE (punti = osservazioni singole, linea nera = media su curve):")
                for g in macro_groups:
                    print(f"  {g}: E0_n={len(e0_points_by_macro[g])}, E0_media_curve={mean_curve_e0[g]:.6g} | betaE_n={len(beta_points_by_macro[g])}, betaE_media_curve={mean_curve_beta[g]:.6g}")

print(f"PDF salvato: {pdf_path}")


Grafico finale E0 + betaE (punti = osservazioni singole, linea nera = media su curve):
  A1: E0_n=155, E0_media_curve=668.322 | betaE_n=155, betaE_media_curve=0.0767321
  0R: E0_n=139, E0_media_curve=1089.92 | betaE_n=139, betaE_media_curve=0.0919906
PDF salvato: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_report_A1_0R.pdf


In [69]:
import os
import pandas as pd

# Usa il CSV del run corrente (quello creato dalla sezione 4 del notebook)
if "path_all" in globals() and isinstance(path_all, str) and os.path.exists(path_all):
    source_csv = path_all
else:
    source_csv = os.path.join(OUTPUT_DIR, "ting_summary_tutte_le_curve_A1_0R.csv")

if not os.path.exists(source_csv):
    raise FileNotFoundError(f"CSV sorgente non trovato: {source_csv}")

out_dir = os.path.dirname(source_csv)
out_csv = os.path.join(out_dir, "boxplot_finale_A1_0R_E0_betaE.csv")

df_box = pd.read_csv(source_csv)
df_box = df_box[df_box["macro_group"].isin(["A1", "0R"])].copy()

for col in ["E0_Pa", "betaE"]:
    if col in df_box.columns:
        df_box[col] = pd.to_numeric(df_box[col], errors="coerce")

# Mantieni solo righe con almeno uno dei due valori del boxplot finale
cols_present = [c for c in ["E0_Pa", "betaE"] if c in df_box.columns]
if not cols_present:
    raise ValueError("Nessuna delle colonne E0_Pa / betaE trovata nel CSV sorgente.")

df_box = df_box[df_box[cols_present].notna().any(axis=1)].copy()

base_cols = ["macro_group", "group"]
for opt in ["cell_folder", "curve"]:
    if opt in df_box.columns:
        base_cols.append(opt)
export_cols = base_cols + cols_present

df_export = df_box[export_cols].sort_values(["macro_group", "group"] + [c for c in ["cell_folder", "curve"] if c in df_box.columns])
df_export.to_csv(out_csv, index=False)

print(f"Sorgente: {source_csv}")
print(f"CSV creato: {out_csv}")
print(f"Righe esportate: {len(df_export)}")
print(df_export[["macro_group"] + cols_present].groupby("macro_group").agg(["count", "median", "mean"]))


Sorgente: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_tutte_le_curve_A1_0R.csv
CSV creato: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/boxplot_finale_A1_0R_E0_betaE.csv
Righe esportate: 294
            E0_Pa                           betaE                    
            count       median         mean count    median      mean
macro_group                                                          
0R            139  1026.498196  1089.924097   139  0.088903  0.091991
A1            155   602.993514   668.321913   155  0.074879  0.076732


In [70]:
# Conteggio punti visualizzati nei boxplot per tipologia (A1/0R) e giornata
for macro_name in ["A1", "0R"]:
    sub = df[df["macro_group"] == macro_name].copy() if "macro_group" in df.columns else pd.DataFrame()
    if len(sub) == 0:
        continue
    print(f"\n=== {macro_name} ===")
    for col in PLOT_PARAMS.keys():
        total_points = int(sub[col].notna().sum()) if col in sub.columns else 0
        print(f"{col}: totale punti = {total_points}")
        by_group = sub.groupby("group")[col].apply(lambda s: int(s.notna().sum())).sort_index()
        print(by_group.to_string())



=== A1 ===
E0_Pa: totale punti = 155
group
100426_A1    58
140326_A1    24
240326_A1    29
270326_A1    44
betaE: totale punti = 155
group
100426_A1    58
140326_A1    24
240326_A1    29
270326_A1    44

=== 0R ===
E0_Pa: totale punti = 139
group
140326_0R    29
150426_0R    45
160426_0R    29
240326_0R    36
betaE: totale punti = 139
group
140326_0R    29
150426_0R    45
160426_0R    29
240326_0R    36
